In [1]:
# ==========================================
# 🚀 WHO'S TALKING — FINAL STABLE VERSION
# ==========================================

import os
import gc
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ==========================================
# ⚙️ CONFIG
# ==========================================
DATA_DIR = '/kaggle/input/competitions/whos-talking-classify-the-app-by-its-packets'

BATCH_SIZE = 512
EPOCHS = 8
N_SPLITS = 3
CHUNK_SIZE = 100000

np.random.seed(42)
tf.random.set_seed(42)

# GPU safe mode
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

# ==========================================
# 📦 LOAD DATA
# ==========================================
dtype_dict = {f'tcp_len_{i}': np.int16 for i in range(1, 31)}

train_df = pd.read_csv(f"{DATA_DIR}/train.csv", dtype=dtype_dict, low_memory=True)
test_df = pd.read_csv(f"{DATA_DIR}/test.csv", dtype=dtype_dict, low_memory=True)

packet_cols = [f'tcp_len_{i}' for i in range(1, 31)]
target_col = 'app_service'

# ==========================================
# 🧠 LABEL ENCODING
# ==========================================
le = LabelEncoder()
y = le.fit_transform(train_df[target_col].astype(str)).astype(np.int32)
num_classes = len(le.classes_)

# ==========================================
# 🔥 PREPROCESS (float16 + numpy)
# ==========================================
def preprocess(df):
    X = df[packet_cols].values.astype(np.float16)

    X = X / 1500.0

    size = np.abs(X)
    direction = np.sign(X)

    X = np.stack([size, direction], axis=-1).astype(np.float16)
    return X

X = preprocess(train_df)
X_test = preprocess(test_df)

test_ids = test_df['id'].values

# освобождаем память
del train_df, test_df
gc.collect()

print("✅ X shape:", X.shape)

# ==========================================
# 🏗️ MODEL (FIXED)
# ==========================================
def create_model():
    inp = keras.Input(shape=(30, 2))

    x = layers.Conv1D(64, 3, padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)

    x = layers.Conv1D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)

    x = layers.Conv1D(256, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)

    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)

    # ✅ ВАЖНО: правильный вызов слоя
    out = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)

    model = keras.Model(inp, out)

    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# ==========================================
# 📦 CHUNKED PREDICT
# ==========================================
def predict_in_chunks(model, X_data):
    preds = []
    n = len(X_data)

    for i in range(0, n, CHUNK_SIZE):
        chunk = X_data[i:i+CHUNK_SIZE]

        p = model.predict(chunk, batch_size=512, verbose=0)
        preds.append(p.astype(np.float16))

        del chunk, p
        gc.collect()

    return np.vstack(preds)

# ==========================================
# 🎯 TRAIN
# ==========================================
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_preds = np.zeros((len(X), num_classes), dtype=np.float16)
test_preds = np.zeros((len(X_test), num_classes), dtype=np.float16)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n===== FOLD {fold}/{N_SPLITS} =====")

    model = create_model()

    model.fit(
        X[train_idx], y[train_idx],
        validation_data=(X[val_idx], y[val_idx]),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=1,
        callbacks=[
            keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(patience=1)
        ]
    )

    # OOF
    val_pred = model.predict(X[val_idx], batch_size=512, verbose=0)
    oof_preds[val_idx] = val_pred.astype(np.float16)

    # TEST
    test_preds += predict_in_chunks(model, X_test) / N_SPLITS

    del model, val_pred
    gc.collect()
    tf.keras.backend.clear_session()

# ==========================================
# 📊 SCORE
# ==========================================
oof_labels = oof_preds.argmax(axis=1)
print("\n📊 OOF F1:", f1_score(y, oof_labels, average='macro'))

# ==========================================
# 📦 SUBMISSION
# ==========================================
test_labels = test_preds.argmax(axis=1)

submission = pd.DataFrame({
    "id": test_ids,
    "app_service": le.inverse_transform(test_labels)
})

submission.to_csv("submission.csv", index=False)
print("✅ submission.csv saved")

2026-04-11 12:24:59.704128: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775910299.916798      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775910299.980248      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775910300.457703      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775910300.457758      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775910300.457761      23 computation_placer.cc:177] computation placer alr

✅ X shape: (8248546, 30, 2)

===== FOLD 1/3 =====


I0000 00:00:1775910408.531900      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Epoch 1/8


I0000 00:00:1775910421.805121      66 service.cc:152] XLA service 0x7aa68c012700 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775910421.805165      66 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1775910422.494278      66 cuda_dnn.cc:529] Loaded cuDNN version 91002


   19/10741 ━━━━━━━━━━━━━━━━━━━━ 1:33 9ms/step - accuracy: 0.0210 - loss: 5.3855

I0000 00:00:1775910427.757336      66 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


10741/10741 ━━━━━━━━━━━━━━━━━━━━ 122s 11ms/step - accuracy: 0.6864 - loss: 1.3067 - val_accuracy: 0.8826 - val_loss: 0.4000 - learning_rate: 0.0010
Epoch 2/8
10741/10741 ━━━━━━━━━━━━━━━━━━━━ 106s 10ms/step - accuracy: 0.8602 - loss: 0.4994 - val_accuracy: 0.9027 - val_loss: 0.3151 - learning_rate: 0.0010
Epoch 3/8
10741/10741 ━━━━━━━━━━━━━━━━━━━━ 107s 10ms/step - accuracy: 0.8769 - loss: 0.4280 - val_accuracy: 0.9069 - val_loss: 0.2999 - learning_rate: 0.0010
Epoch 4/8
10741/10741 ━━━━━━━━━━━━━━━━━━━━ 108s 10ms/step - accuracy: 0.8858 - loss: 0.3928 - val_accuracy: 0.9121 - val_loss: 0.2790 - learning_rate: 0.0010
Epoch 5/8
10741/10741 ━━━━━━━━━━━━━━━━━━━━ 107s 10ms/step - accuracy: 0.8914 - loss: 0.3698 - val_accuracy: 0.9171 - val_loss: 0.2597 - learning_rate: 0.0010
Epoch 6/8
10741/10741 ━━━━━━━━━━━━━━━━━━━━ 106s 10ms/step - accuracy: 0.8955 - loss: 0.3538 - val_accuracy: 0.9204 - val_loss: 0.2467 - learning_rate: 0.0010
Epoch 7/8
10741/10741 ━━━━━━━━━━━━━━━━━━━━ 105s 10ms/step - ac